# 威尔科克森秩和检验完整教程：非参数两样本检验

## 📚 教学目标
1. 理解非参数检验的优势和适用场景
2. 掌握秩和检验的完整步骤
3. 手算秩次和检验统计量
4. 用 scipy.stats.ranksums 验证手算结果
5. 对比秩和检验与 t 检验的结果

**参考书**：《基础统计学(第14版)》（Triola）第 13-4 节
**教学策略**：先理解为什么需要非参数检验，再通过手算掌握秩和检验的原理

---

## 1. 场景设定：为什么需要非参数检验？

### 🎯 问题

比较两种教学方法对学生考试成绩的影响：
- 方法 A: 12 名学生
- 方法 B: 10 名学生

问题：两种教学方法的效果是否相同？

### 📖 书中的定义

> 非参数检验（nonparametric test）不依赖于总体分布的假设。
> 当数据不满足参数检验的条件（如正态性）时，应使用非参数方法。

### 💡 参数检验 vs 非参数检验

| 特性 | 参数检验 | 非参数检验 |
|------|----------|------------|
| 分布假设 | 需要正态性 | 不需要 |
| 数据类型 | 连续变量 | 连续或有序 |
| 检验功效 | 高（满足条件时）| 较低 |
| 异常值 | 敏感 | 稳健 |
| 样本量 | 大样本 | 小样本也适用 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据手算

### 📐 秩和检验步骤

1. 合并两组数据
2. 按大小排序，分配秩次
3. 计算较小样本的秩和 W
4. 计算检验统计量 z
5. 查表或计算 p 值

In [ ]:
# ========== 步骤 1: 生成数据 ==========

print("=" * 60)
print("📋 威尔科克森秩和检验: 教学方法比较")
print("=" * 60)

# --- 数据生成过程 (DGP) ---
# 方法 A: 传统教学 (均值较低)
method_a = np.array([65, 70, 72, 68, 75, 71, 69, 73, 67, 74, 70, 72])

# 方法 B: 新教学法 (均值较高)
method_b = np.array([78, 82, 76, 80, 85, 77, 83, 79, 81, 84])

print(f"\n📊 数据概况:")
print(f"  方法 A: n = {len(method_a)}, 均值 = {np.mean(method_a):.2f}")
print(f"  方法 B: n = {len(method_b)}, 均值 = {np.mean(method_b):.2f}")

# 合并数据并排序
combined = np.concatenate([method_a, method_b])
group_labels = ['A'] * len(method_a) + ['B'] * len(method_b)

# 排序并分配秩次
sorted_indices = np.argsort(combined)
sorted_data = combined[sorted_indices]
sorted_labels = [group_labels[i] for i in sorted_indices]

# 处理并列值 (平均秩次)
ranks = np.zeros(len(combined))
i = 0
while i < len(sorted_data):
    j = i
    while j < len(sorted_data) and sorted_data[j] == sorted_data[i]:
        j += 1
    avg_rank = (i + j + 1) / 2  # 平均秩次
    for k in range(i, j):
        ranks[sorted_indices[k]] = avg_rank
    i = j

print(f"\n📊 步骤 1: 合并排序并分配秩次")
print(f"{'值':<8} {'组别':<6} {'秩次':<8}")
print("-" * 22)
for idx in sorted_indices:
    print(f"  {combined[idx]:<8} {group_labels[idx]:<6} {ranks[idx]:<8.1f}")

# 计算秩和
rank_sum_a = np.sum(ranks[:len(method_a)])
rank_sum_b = np.sum(ranks[len(method_a):])

print(f"\n📊 步骤 2: 计算秩和")
print(f"  方法 A 的秩和 W_A = {rank_sum_a:.1f}")
print(f"  方法 B 的秩和 W_B = {rank_sum_b:.1f}")

# 使用较小样本的秩和
n1 = min(len(method_a), len(method_b))
n2 = max(len(method_a), len(method_b))
W = rank_sum_b if len(method_b) <= len(method_a) else rank_sum_a

print(f"\n📊 步骤 3: 检验统计量")
print(f"  n1 = {n1}, n2 = {n2}")
print(f"  W = {W:.1f}")

print("\n" + "=" * 60)

In [ ]:
# ========== 步骤 2: scipy 验证 ==========

print("=" * 60)
print("🔬 scipy.stats.ranksums 验证")
print("=" * 60)

z_scipy, p_scipy = stats.ranksums(method_b, method_a)

print(f"\n📊 scipy 结果:")
print(f"  z 统计量 = {z_scipy:.4f}")
print(f"  p 值 = {p_scipy:.6f}")

# 判断
alpha = 0.05
if p_scipy < alpha:
    print(f"\n  💡 p < {alpha}，拒绝 H₀：两种教学方法的效果不同")
else:
    print(f"\n  💡 p ≥ {alpha}，不能拒绝 H₀")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 秩和检验 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱形图对比
ax1 = axes[0]
bp = ax1.boxplot([method_a, method_b], labels=['Method A', 'Method B'],
                patch_artist=True)
colors = ['#2ecc71', '#3498db']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Score Distribution by Method', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 秩次分布
ax2 = axes[1]
all_ranks_a = ranks[:len(method_a)]
all_ranks_b = ranks[len(method_a):]
ax2.hist(all_ranks_a, bins=10, alpha=0.6, color='#2ecc71', label='Method A', edgecolor='white')
ax2.hist(all_ranks_b, bins=10, alpha=0.6, color='#3498db', label='Method B', edgecolor='white')
ax2.set_xlabel('Rank', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Ranks', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：两组数据的箱形图")
print("  右图：两组数据的秩次分布")
print("  如果两组效果相同，秩次应均匀分布在两组中")
print("  方法 B 的秩次明显偏高，说明其效果更好")

---

## 3. 参数检验 vs 非参数检验对比

### 💡 核心对比

我们同时进行 t 检验和秩和检验，比较结果。

In [ ]:
# ========== 步骤 3: 参数 vs 非参数对比 ==========

print("=" * 60)
print("📋 参数检验 vs 非参数检验对比")
print("=" * 60)

# t 检验
t_stat, t_p = stats.ttest_ind(method_a, method_b)

# 秩和检验
w_stat, w_p = stats.ranksums(method_a, method_b)

print(f"\n📊 检验结果对比:")
print(f"{'检验方法':<20} {'统计量':<15} {'p 值':<15}")
print("-" * 50)
print(f"  {'t 检验':<18} {'t = ' + f'{t_stat:.4f}':<15} {t_p:<15.6f}")
print(f"  {'秩和检验':<16} {'z = ' + f'{w_stat:.4f}':<15} {w_p:<15.6f}")

alpha = 0.05
print(f"\n📊 判断 (α = {alpha}):")
print(f"  t 检验: {'拒绝 H₀' if t_p < alpha else '不能拒绝 H₀'}")
print(f"  秩和检验: {'拒绝 H₀' if w_p < alpha else '不能拒绝 H₀'}")

print(f"\n💡 解读:")
print(f"  当数据满足正态性时，t 检验的功效更高")
print(f"  当数据不满足正态性时，秩和检验更稳健")
print(f"  两种方法通常给出一致的结论")

print("\n" + "=" * 60)

---

## 4. 大样本示例

### 🎯 问题

比较两种药物的疗效，样本量较大（每组 50 人）。

In [ ]:
# ========== 步骤 4: 大样本示例 ==========

print("=" * 60)
print("📋 大样本秩和检验: 药物疗效比较")
print("=" * 60)

# --- 数据生成过程 (DGP) ---
n_large = 50
drug_x = np.random.normal(50, 10, n_large)
drug_y = np.random.normal(55, 10, n_large)

print(f"\n📊 数据概况:")
print(f"  药物 X: n = {n_large}, 均值 = {np.mean(drug_x):.2f}, 中位数 = {np.median(drug_x):.2f}")
print(f"  药物 Y: n = {n_large}, 均值 = {np.mean(drug_y):.2f}, 中位数 = {np.median(drug_y):.2f}")

# 正态性检验
print(f"\n📊 正态性检验:")
for name, data in [('X', drug_x), ('Y', drug_y)]:
    w_stat, p_val = stats.shapiro(data)
    result = '正态' if p_val > 0.05 else '非正态'
    print(f"  药物 {name}: W = {w_stat:.4f}, p = {p_val:.4f} ({result})")

# t 检验
t_stat_large, t_p_large = stats.ttest_ind(drug_x, drug_y)

# 秩和检验
z_stat_large, z_p_large = stats.ranksums(drug_x, drug_y)

print(f"\n📊 检验结果:")
print(f"  t 检验: t = {t_stat_large:.4f}, p = {t_p_large:.6f}")
print(f"  秩和检验: z = {z_stat_large:.4f}, p = {z_p_large:.6f}")

alpha = 0.05
print(f"\n📊 判断 (α = {alpha}):")
print(f"  t 检验: {'拒绝 H₀' if t_p_large < alpha else '不能拒绝 H₀'}")
print(f"  秩和检验: {'拒绝 H₀' if z_p_large < alpha else '不能拒绝 H₀'}")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化: 大样本对比 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱形图
ax1 = axes[0]
bp = ax1.boxplot([drug_x, drug_y], labels=['Drug X', 'Drug Y'],
                patch_artist=True)
colors = ['#2ecc71', '#3498db']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Efficacy Score', fontsize=12)
ax1.set_title('Drug Efficacy Comparison', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 直方图对比
ax2 = axes[1]
ax2.hist(drug_x, bins=15, alpha=0.6, color='#2ecc71', label='Drug X', density=True, edgecolor='white')
ax2.hist(drug_y, bins=15, alpha=0.6, color='#3498db', label='Drug Y', density=True, edgecolor='white')
ax2.set_xlabel('Efficacy Score', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Distribution Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：两组数据的箱形图")
print("  右图：两组数据的密度直方图")
print("  大样本时，t 检验和秩和检验通常给出一致结论")

---

## 📌 核心概念回顾

### 📌 威尔科克森秩和检验 (Wilcoxon Rank-Sum Test)
- **定义**: 检验两个独立样本是否来自同一总体
- **原理**: 将数据转换为秩次，比较两组的秩和
- **含义**: 如果两组效果相同，秩次应均匀分布
- **判断标准**: p < α 时拒绝 H₀

### 📌 秩次 (Rank)
- **定义**: 数据按大小排序后的位置编号
- **处理并列值**: 使用平均秩次
- **优势**: 不受异常值影响

### 📌 参数 vs 非参数检验
- **参数检验**: 依赖分布假设，功效高
- **非参数检验**: 不依赖分布假设，功效较低
- **选择依据**: 数据是否满足正态性

### 📌 适用场景
- 数据严重偏离正态分布
- 样本量很小
- 数据是有序变量（如 Likert 量表）
- 存在异常值

### 🔗 完整流程
```
合并数据 → 排序赋秩 → 计算秩和 → 计算 z → 求 p 值 → 判断
    ↓          ↓          ↓        ↓        ↓       ↓
  两组数据   平均秩次    W 统计量   公式    查表/软件  p<α?
```

---

## ❌ 常见误区

### ❌ 误区 1: 所有数据都应该用非参数检验
**✓ 正确做法**: 当数据满足正态性时，参数检验（如 t 检验）的功效更高。只有当数据不满足正态性假设时，才应使用非参数检验。不要"为了安全"而盲目使用非参数方法。

### ❌ 误区 2: 在不满足参数检验条件时误用参数方法
**✓ 正确做法**: 当数据严重偏离正态分布或存在异常值时，应使用非参数检验。强行使用 t 检验可能导致错误的结论。应先检验正态性（如 Shapiro-Wilk），再选择合适的检验方法。

### ❌ 误区 3: 非参数检验没有假设
**✓ 正确理解**: 非参数检验也有假设，如独立性、随机抽样等。秩和检验还假设两组数据的分布形状相同（只是位置不同）。如果分布形状不同，秩和检验可能不适用。

### ❌ 误区 4: 秩和检验的 p 值与 t 检验的 p 值应该相同
**✓ 正确理解**: 两种检验的 p 值通常不同，因为它们检验的假设不同。t 检验检验均值差异，秩和检验检验分布位置差异。当数据正态时，两者通常给出一致的结论。

### ❌ 误区 5: 非参数检验可以完全替代参数检验
**✓ 正确理解**: 非参数检验的功效（检测真实差异的能力）通常低于参数检验。当数据满足正态性时，使用非参数检验会浪费信息，降低检验功效。应根据数据特征选择合适的检验方法。